# Predicting Time to the Next FDA Inspection with Survival Curves

This notebook is a self-contained, synthetic example of how survival analysis can estimate **when a site may receive its next FDA inspection**.

It demonstrates:

- how censoring differs from simply dropping sites without a later inspection;
- how to read a Kaplan–Meier survival curve;
- how to convert survival probabilities into inspection probabilities;
- how curves can differ by product type;
- how hazard ratios quantify relative inspection rates;
- how to structure the real inspection data without introducing temporal leakage.

> The generated observations are illustrative and are not actual FDA predictions.

## 1. What survival analysis predicts

For a site inspected at time zero, define:

- $T$: number of days until its next inspection;
- $S(t) = P(T > t)$: probability the site has **not yet** been inspected again by day $t$;
- $1-S(t)$: probability the site **has** been inspected again by day $t$.

A site with no later inspection in the available data is not necessarily a permanent non-event. Its waiting time is **right-censored** at the end of observation.

This makes survival analysis more appropriate than ordinary regression for time-to-next-inspection data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(value: object) -> None:
        print(value)
from matplotlib.ticker import PercentFormatter


RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


QUALIFYZE_COLORS = {
    "primary": "#618FFC",
    "pale": "#EEF3FF",
    "risk": "#E34A3A",
    "navy": "#33374D",
    "teal": "#4FAF9F",
    "purple": "#9B78C6",
    "grid": "#DCE6FF",
    "background": "#FFFFFF",
}


plt.rcParams.update(
    {
        "figure.facecolor": QUALIFYZE_COLORS["background"],
        "axes.facecolor": QUALIFYZE_COLORS["background"],
        "axes.edgecolor": QUALIFYZE_COLORS["navy"],
        "axes.labelcolor": QUALIFYZE_COLORS["navy"],
        "text.color": QUALIFYZE_COLORS["navy"],
        "xtick.color": QUALIFYZE_COLORS["navy"],
        "ytick.color": QUALIFYZE_COLORS["navy"],
        "font.size": 11,
    }
)

## 2. Generate an illustrative inspection-history dataset

Each row represents one site after an index inspection. The event is the next inspection. The synthetic product types have different average waiting times so their survival curves are visibly different.

The observation window ends before every site receives another inspection, creating realistic right-censored records.

In [ ]:
PRODUCT_TYPES = np.array(
    [
        "Food",
        "Devices",
        "Drugs",
        "Biologics",
    ]
)

PRODUCT_PROBABILITIES = np.array(
    [
        0.36,
        0.28,
        0.24,
        0.12,
    ]
)

# Used only to generate the synthetic example.
# Shorter mean waiting time means a higher inspection hazard.
SIMULATED_MEAN_WAIT_DAYS = {
    "Food": 1_050,
    "Devices": 820,
    "Drugs": 650,
    "Biologics": 520,
}


number_of_sites = 2_400

product_type = rng.choice(
    PRODUCT_TYPES,
    size=number_of_sites,
    p=PRODUCT_PROBABILITIES,
)

latent_event_time = np.array(
    [
        rng.exponential(
            SIMULATED_MEAN_WAIT_DAYS[product]
        )
        for product in product_type
    ]
)

# Facilities enter the dataset at different dates, so their
# available follow-up differs.
available_follow_up = rng.uniform(
    730,
    1_095,
    size=number_of_sites,
)

duration_days = np.ceil(
    np.minimum(
        latent_event_time,
        available_follow_up,
    )
).astype(int)

event_observed = (
    latent_event_time <= available_follow_up
).astype(int)


survival_data = pd.DataFrame(
    {
        "site_id": np.arange(
            1,
            number_of_sites + 1,
        ),
        "product_type": product_type,
        "duration_days": duration_days,
        "event_observed": event_observed,
    }
)


display(survival_data.head())

display(
    survival_data.groupby(
        "product_type"
    ).agg(
        facilities=("site_id", "size"),
        next_inspections=(
            "event_observed",
            "sum",
        ),
        censored=(
            "event_observed",
            lambda values: int(
                (values == 0).sum()
            ),
        ),
        median_observed_days=(
            "duration_days",
            "median",
        ),
    )
)

## 3. A small Kaplan–Meier implementation

At each event time, the Kaplan–Meier estimator multiplies the previous survival probability by:

$$
1 - \frac{d_t}{n_t}
$$

where:

- $d_t$ is the number of next inspections at time $t$;
- $n_t$ is the number of sites still at risk immediately before $t$.

Censored sites remain in the risk set until their censoring date, but are never counted as inspection events.

In [ ]:
def kaplan_meier_curve(
    durations: pd.Series | np.ndarray,
    events: pd.Series | np.ndarray,
) -> pd.DataFrame:
    durations_array = np.asarray(
        durations,
        dtype=float,
    )
    events_array = np.asarray(
        events,
        dtype=bool,
    )

    event_times = np.sort(
        np.unique(
            durations_array[events_array]
        )
    )

    survival_probability = 1.0

    rows = [
        {
            "time": 0.0,
            "at_risk": len(durations_array),
            "events": 0,
            "survival_probability": 1.0,
        }
    ]

    for event_time in event_times:
        at_risk = int(
            np.sum(
                durations_array >= event_time
            )
        )

        event_count = int(
            np.sum(
                (durations_array == event_time)
                & events_array
            )
        )

        survival_probability *= (
            1.0 - event_count / at_risk
        )

        rows.append(
            {
                "time": float(event_time),
                "at_risk": at_risk,
                "events": event_count,
                "survival_probability":
                    survival_probability,
            }
        )

    return pd.DataFrame(rows)


def survival_probability_at(
    curve: pd.DataFrame,
    horizon_days: int,
) -> float:
    available = curve.loc[
        curve["time"] <= horizon_days,
        "survival_probability",
    ]

    if available.empty:
        return 1.0

    return float(available.iloc[-1])


def median_survival_time(
    curve: pd.DataFrame,
) -> float:
    below_half = curve.loc[
        curve["survival_probability"] <= 0.5,
        "time",
    ]

    if below_half.empty:
        return np.nan

    return float(below_half.iloc[0])

## 4. Overall time-to-next-inspection curve

The curve starts at 100% because no site has yet received its next inspection. It falls whenever another inspection occurs.

For example, if $S(365)=0.70$, then:

- approximately 70% of sites remain without a new inspection after one year;
- approximately 30% have been inspected again within one year.

In [ ]:
overall_curve = kaplan_meier_curve(
    survival_data["duration_days"],
    survival_data["event_observed"],
)


prediction_horizons = [
    180,
    365,
    730,
    1_095,
]

overall_predictions = pd.DataFrame(
    {
        "horizon_days": prediction_horizons,
        "probability_no_inspection": [
            survival_probability_at(
                overall_curve,
                horizon,
            )
            for horizon in prediction_horizons
        ],
    }
)

overall_predictions[
    "probability_inspected"
] = (
    1
    - overall_predictions[
        "probability_no_inspection"
    ]
)


overall_predictions_display = (
    overall_predictions.copy()
)

for column in (
    "probability_no_inspection",
    "probability_inspected",
):
    overall_predictions_display[column] = (
        overall_predictions_display[column].map(
            lambda value: f"{value:.1%}"
        )
    )

display(overall_predictions_display)


overall_median = median_survival_time(
    overall_curve
)

print(
    "Estimated median time to the next inspection:",
    (
        f"{overall_median:,.0f} days"
        if np.isfinite(overall_median)
        else "not reached during follow-up"
    ),
)


fig, ax = plt.subplots(
    figsize=(12, 6),
    constrained_layout=True,
)

ax.step(
    overall_curve["time"],
    overall_curve["survival_probability"],
    where="post",
    color=QUALIFYZE_COLORS["primary"],
    linewidth=3,
)

ax.fill_between(
    overall_curve["time"],
    overall_curve["survival_probability"],
    step="post",
    color=QUALIFYZE_COLORS["pale"],
    alpha=0.8,
)

for horizon in (365, 730):
    probability = survival_probability_at(
        overall_curve,
        horizon,
    )

    ax.axvline(
        horizon,
        color=QUALIFYZE_COLORS["grid"],
        linestyle="--",
        linewidth=1,
    )

    ax.annotate(
        f"{probability:.1%} remain uninspected",
        xy=(horizon, probability),
        xytext=(8, 10),
        textcoords="offset points",
        color=QUALIFYZE_COLORS["navy"],
    )

ax.set_ylim(0, 1.03)
ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=1)
)
ax.set_xlabel(
    "Days since the previous inspection"
)
ax.set_ylabel(
    "Probability of no new inspection"
)
ax.set_title(
    "Estimated time until the next inspection"
)
ax.grid(
    axis="y",
    color=QUALIFYZE_COLORS["grid"],
    alpha=0.7,
)

plt.show()

## 5. Survival curves by product type

Separate curves allow the estimated waiting-time distribution to vary across a business-relevant characteristic.

A curve that falls faster indicates that sites in that product category tend to receive their next inspection sooner. The vertical distance between two curves at a chosen horizon is an absolute probability difference.

In [ ]:
PRODUCT_COLORS = {
    "Food": QUALIFYZE_COLORS["primary"],
    "Devices": QUALIFYZE_COLORS["teal"],
    "Drugs": QUALIFYZE_COLORS["risk"],
    "Biologics": QUALIFYZE_COLORS["purple"],
}


product_curves: dict[str, pd.DataFrame] = {}
product_summaries: list[dict[str, object]] = []


fig, ax = plt.subplots(
    figsize=(12, 6.5),
    constrained_layout=True,
)


for product in PRODUCT_TYPES:
    group = survival_data.loc[
        survival_data["product_type"]
        == product
    ]

    curve = kaplan_meier_curve(
        group["duration_days"],
        group["event_observed"],
    )

    product_curves[product] = curve

    median_days = median_survival_time(
        curve
    )

    product_summaries.append(
        {
            "product_type": product,
            "facilities": len(group),
            "median_days_to_next_inspection":
                median_days,
            "probability_inspected_within_1_year":
                1
                - survival_probability_at(
                    curve,
                    365,
                ),
            "probability_inspected_within_2_years":
                1
                - survival_probability_at(
                    curve,
                    730,
                ),
        }
    )

    ax.step(
        curve["time"],
        curve["survival_probability"],
        where="post",
        linewidth=2.5,
        color=PRODUCT_COLORS[product],
        label=product,
    )


ax.set_ylim(0, 1.03)
ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=1)
)
ax.set_xlabel(
    "Days since the previous inspection"
)
ax.set_ylabel(
    "Probability of no new inspection"
)
ax.set_title(
    "Time to next inspection varies by product type"
)
ax.grid(
    axis="y",
    color=QUALIFYZE_COLORS["grid"],
    alpha=0.7,
)
ax.legend(
    title="Product type",
    frameon=False,
)

plt.show()


product_summary = pd.DataFrame(
    product_summaries
).sort_values(
    "median_days_to_next_inspection"
)


product_summary_display = product_summary.copy()

product_summary_display[
    "median_days_to_next_inspection"
] = product_summary_display[
    "median_days_to_next_inspection"
].map(
    lambda value: f"{value:,.0f}"
)

for column in (
    "probability_inspected_within_1_year",
    "probability_inspected_within_2_years",
):
    product_summary_display[column] = (
        product_summary_display[column].map(
            lambda value: f"{value:.1%}"
        )
    )

display(product_summary_display)

### How to communicate the product curves

Suppose the two-year inspection probabilities are 75% for Biologics and 52% for Food. At that horizon, the estimated absolute difference is 23 percentage points.

This comparison is easy to communicate, but it is **unadjusted**: other differences between those facilities could partly explain the curves. A fitted survival regression model can adjust for inspection history, citations, prior actions, geography and other variables.

## 6. Hazard ratios: the survival-model equivalent of relative risk

In a proportional-hazards model:

$$
h(t \mid x) = h_0(t)\exp(\beta x)
$$

and:

$$
HR = \exp(\beta)
$$

For a product category relative to a reference category:

- HR = 1.00: the instantaneous inspection rate is the same;
- HR = 1.50: the category has a 50% higher instantaneous inspection hazard;
- HR = 0.70: the category has a 30% lower instantaneous inspection hazard.

A hazard ratio is **not** a probability ratio and does not mean that an inspection occurs exactly 50% sooner.

The following calculation is deliberately simple: it estimates an event rate using inspections divided by observed site-time. A production model should estimate adjusted hazard ratios with a Cox model or another survival model.

In [ ]:
hazard_summary = (
    survival_data.groupby(
        "product_type",
        as_index=False,
    )
    .agg(
        facilities=("site_id", "size"),
        next_inspections=(
            "event_observed",
            "sum",
        ),
        observed_site_days=(
            "duration_days",
            "sum",
        ),
    )
)


hazard_summary[
    "inspection_hazard_per_site_year"
] = (
    hazard_summary["next_inspections"]
    / hazard_summary["observed_site_days"]
    * 365.25
)


REFERENCE_PRODUCT = "Food"

reference_hazard = float(
    hazard_summary.loc[
        hazard_summary["product_type"]
        == REFERENCE_PRODUCT,
        "inspection_hazard_per_site_year",
    ].iloc[0]
)


hazard_summary["hazard_ratio"] = (
    hazard_summary[
        "inspection_hazard_per_site_year"
    ]
    / reference_hazard
)

hazard_summary[
    "log_hazard_ratio"
] = np.log(
    hazard_summary["hazard_ratio"]
)


hazard_summary_display = (
    hazard_summary.sort_values(
        "hazard_ratio",
        ascending=False,
    ).copy()
)

hazard_summary_display[
    "inspection_hazard_per_site_year"
] = hazard_summary_display[
    "inspection_hazard_per_site_year"
].map(lambda value: f"{value:.3f}")

hazard_summary_display[
    "hazard_ratio"
] = hazard_summary_display[
    "hazard_ratio"
].map(lambda value: f"{value:.2f}")

hazard_summary_display[
    "log_hazard_ratio"
] = hazard_summary_display[
    "log_hazard_ratio"
].map(lambda value: f"{value:+.3f}")

display(hazard_summary_display)


hazard_plot = hazard_summary.sort_values(
    "hazard_ratio"
).copy()

hazard_colors = np.where(
    hazard_plot["hazard_ratio"] >= 1,
    QUALIFYZE_COLORS["risk"],
    QUALIFYZE_COLORS["primary"],
)

hazard_colors = np.where(
    hazard_plot["product_type"]
    == REFERENCE_PRODUCT,
    QUALIFYZE_COLORS["navy"],
    hazard_colors,
)


fig, ax = plt.subplots(
    figsize=(11, 5.5),
    constrained_layout=True,
)

bars = ax.bar(
    hazard_plot["product_type"],
    hazard_plot["hazard_ratio"],
    color=hazard_colors,
    alpha=0.9,
    width=0.65,
)

ax.axhline(
    1,
    color=QUALIFYZE_COLORS["navy"],
    linestyle="--",
    linewidth=1.2,
)

for bar, value in zip(
    bars,
    hazard_plot["hazard_ratio"],
):
    ax.annotate(
        f"{value:.2f}×",
        xy=(
            bar.get_x()
            + bar.get_width() / 2,
            value,
        ),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=11,
    )

maximum_ratio = float(
    hazard_plot["hazard_ratio"].max()
)

ax.set_ylim(
    0,
    maximum_ratio * 1.20,
)
ax.set_xlabel("Product type")
ax.set_ylabel(
    f"Hazard ratio relative to {REFERENCE_PRODUCT}"
)
ax.set_title(
    "Relative rate of receiving the next inspection"
)
ax.grid(
    axis="y",
    color=QUALIFYZE_COLORS["grid"],
    alpha=0.7,
)

plt.show()

## 7. How hazard ratios change a baseline survival curve

Under the proportional-hazards assumption, a category-specific survival curve can be expressed as:

$$
S(t \mid x) = S_0(t)^{HR}
$$

The next chart uses the Food curve as the illustrative baseline. A hazard ratio above one pulls the survival curve down, indicating shorter waiting times.

In [ ]:
baseline_curve = product_curves[
    REFERENCE_PRODUCT
][
    [
        "time",
        "survival_probability",
    ]
].copy()


fig, ax = plt.subplots(
    figsize=(12, 6.5),
    constrained_layout=True,
)


for row in hazard_summary.itertuples(
    index=False
):
    adjusted_survival = (
        baseline_curve[
            "survival_probability"
        ]
        ** row.hazard_ratio
    )

    ax.step(
        baseline_curve["time"],
        adjusted_survival,
        where="post",
        linewidth=2.5,
        color=PRODUCT_COLORS[
            row.product_type
        ],
        label=(
            f"{row.product_type} "
            f"(HR={row.hazard_ratio:.2f})"
        ),
    )


ax.set_ylim(0, 1.03)
ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=1)
)
ax.set_xlabel(
    "Days since the previous inspection"
)
ax.set_ylabel(
    "Probability of no new inspection"
)
ax.set_title(
    "Illustrative product curves implied by hazard ratios"
)
ax.grid(
    axis="y",
    color=QUALIFYZE_COLORS["grid"],
    alpha=0.7,
)
ax.legend(
    frameon=False,
)

plt.show()

## 8. How the real inspection dataset should be structured

For every physical inspection, create an interval that begins at that inspection and ends at either:

1. the next inspection for the same FEI (event_observed = 1); or
2. the data-extraction cutoff (event_observed = 0).

The last known inspection for a facility should normally be retained as a censored observation rather than deleted.

An illustrative PostgreSQL query is shown below:

~~~sql
WITH inspection_events AS (
    SELECT
        inspection_id,
        btrim(fei_number::text) AS fei_number,
        MAX(inspection_end_date) AS inspection_date,
        MAX(NULLIF(btrim(product_type), '')) AS product_type
    FROM public.inspections
    WHERE inspection_id IS NOT NULL
      AND fei_number IS NOT NULL
      AND btrim(fei_number::text) <> ''
      AND inspection_end_date IS NOT NULL
    GROUP BY
        inspection_id,
        btrim(fei_number::text)
),

ordered_inspections AS (
    SELECT
        *,
        LEAD(inspection_date) OVER (
            PARTITION BY fei_number
            ORDER BY inspection_date, inspection_id
        ) AS next_inspection_date
    FROM inspection_events
)

SELECT
    inspection_id,
    fei_number,
    inspection_date AS interval_start_date,
    next_inspection_date,
    product_type,

    CASE
        WHEN next_inspection_date <= :data_cutoff
            THEN next_inspection_date - inspection_date
        ELSE :data_cutoff - inspection_date
    END AS duration_days,

    CASE
        WHEN next_inspection_date <= :data_cutoff
            THEN 1
        ELSE 0
    END AS event_observed

FROM ordered_inspections

WHERE inspection_date < :data_cutoff;
~~~

If one physical inspection contains several product types, define a reproducible rule: retain multiple binary product indicators, select a primary product type, or use a multi-state model. Avoid choosing an arbitrary value with MAX() in a production dataset.

## 9. Features and leakage controls

Candidate predictors measured at the interval start include:

- classification of the current and previous inspections;
- number and rate of prior citations;
- repeated CFR provisions;
- prior published 483s;
- warning letters, injunctions and seizures;
- recall-event history;
- product categories;
- state and country;
- time since the preceding inspection.

Do not use information created after the interval start. In particular, citations or classification from the future inspection are outcomes of the waiting period, not predictors.

A temporal test set is still required: train on earlier intervals and evaluate on later intervals.

## 10. Business interpretation

A survival model can provide several useful outputs for each site:

- probability of another inspection within 6, 12 or 24 months;
- probability of remaining uninspected through a chosen date;
- median predicted waiting time, when the curve crosses 50%;
- relative inspection hazard associated with a product type or historical signal;
- a ranked list of sites for planning and readiness activities.

The model predicts the timing of regulatory attention—not whether a facility is compliant. Hazard ratios describe associations and should not be interpreted as causal effects.